# FixNext — IBM Granite on Colab (T4 GPU)

**Part 1 (cells 1–9): embeddings.** `ibm-granite/granite-embedding-small-english-r2` embeds all 180,308 CVE descriptions (2024–2026) → `granite_embeddings.npy` (180,308 × 384, L2-normalised, float16) + `granite_embeddings_ids.csv`. Used by `src/train.py` for the nearest-known-exploited-CVE similarity features (`gr_knn_*`) and the PCA ablation.

**Part 2 (cells 10–24): rationales.** `ibm-granite/granite-3.3-2b-instruct` writes a two-line "why fix this next" for each top-100 CVE → `outputs/rationales.csv`, `outputs/rationales_2.csv`. Some cells are repeated from prompt iterations; the final prompt is in the last `why_fix_first` cell.

Author: Calvin · Sheridan Datathon 2026 · IBM track "Before the Breach"


In [1]:
!pip -q install -U sentence-transformers

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import time

PATH = '/content/drive/MyDrive/FixFirst/descriptions_2024_2026.csv'
df = pd.read_csv(PATH, dtype=str)

print(df.shape)
print(df.head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(180308, 2)
           cve_id                                        description
0  CVE-2002-20002  The Net::EasyTCP package before 0.15 for Perl ...
1  CVE-2003-20001  An issue was discovered on Mitel ICP VoIP 3100...
2  CVE-2005-10003  A vulnerability classified as critical has bee...


In [5]:
from pathlib import Path

matches = list(Path('/content/drive/MyDrive').rglob('*descriptions*'))
for path in matches:
    print(path)

if not matches:
    print("No matching file found in My Drive.")

/content/drive/MyDrive/FixFirst/descriptions_2024_2026.csv


In [6]:
from sentence_transformers import SentenceTransformer
import torch

assert torch.cuda.is_available(), "GPU not enabled. Select Runtime → Change runtime type → T4 GPU."

MODEL = 'ibm-granite/granite-embedding-small-english-r2'
model = SentenceTransformer(MODEL, device='cuda')
model.max_seq_length = 512

print('GPU:', torch.cuda.get_device_name(0))
print('device:', model.device)
print('embedding dimension:', model.get_sentence_embedding_dimension())
print('missing descriptions:', df.description.isna().sum())

modules.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/12.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 95.3MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

GPU: Tesla T4
device: cuda:0
embedding dimension: 384
missing descriptions: 0


/tmp/ipykernel_715/2484004137.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('embedding dimension:', model.get_sentence_embedding_dimension())


In [7]:
sample = df.description.head(1000).tolist()

t0 = time.perf_counter()
e = model.encode(
    sample,
    batch_size=128,
    normalize_embeddings=True,
    show_progress_bar=True
)
elapsed = time.perf_counter() - t0

print("Sample shape:", e.shape)
print("1,000 rows took:", round(elapsed, 1), "seconds")
print("Estimated full run:", round(elapsed * len(df) / 1000 / 60, 1), "minutes")

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Sample shape: (1000, 384)
1,000 rows took: 4.7 seconds
Estimated full run: 14.0 minutes


In [8]:
from pathlib import Path

OUT = Path('/content/drive/MyDrive/FixFirst')
OUT.mkdir(parents=True, exist_ok=True)

# Save the IDs in exactly the same order as the embeddings.
df[['cve_id']].to_csv(OUT / 'granite_embeddings_ids.csv', index=False)

chunks = []
t0 = time.perf_counter()

for start in range(0, len(df), 10000):
    end = min(start + 10000, len(df))
    checkpoint = OUT / f'granite_chunk_{start}_{end}.npy'

    if checkpoint.exists():
        chunk = np.load(checkpoint)
        assert chunk.shape == (end - start, 384)
        print(f"Loaded checkpoint: {start:,}–{end:,}")
    else:
        chunk = model.encode(
            df.description.iloc[start:end].tolist(),
            batch_size=128,
            normalize_embeddings=True,
            show_progress_bar=True
        ).astype(np.float16)

        np.save(checkpoint, chunk)
        print(f"Saved: {end:,}/{len(df):,}")

    chunks.append(chunk)

emb = np.concatenate(chunks, axis=0)
np.save(OUT / 'granite_embeddings.npy', emb)

elapsed_minutes = (time.perf_counter() - t0) / 60
print("Final shape:", emb.shape)
print("Elapsed minutes:", round(elapsed_minutes, 1))
print("Saved to:", OUT)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Saved: 10,000/180,308


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Saved: 20,000/180,308


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Saved: 30,000/180,308


... (28 outputs left out)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Saved: 180,000/180,308


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Saved: 180,308/180,308
Final shape: (180308, 384)
Elapsed minutes: 19.1
Saved to: /content/drive/MyDrive/FixFirst


In [9]:
saved = np.load(OUT / 'granite_embeddings.npy')
ids = pd.read_csv(OUT / 'granite_embeddings_ids.csv', dtype=str)

assert saved.shape == (len(df), 384)
assert ids.cve_id.tolist() == df.cve_id.tolist()
assert np.isfinite(saved).all()
print("Saved files passed shape, ID alignment and finite-number checks.")

emb32 = saved.astype(np.float32)
i = np.flatnonzero(
    df.description.str.contains('Fortinet|FortiOS', case=False, na=False)
)[0]

sims = emb32 @ emb32[i]
top = np.argsort(-sims)[:6]

for j in top:
    print(
        round(float(sims[j]), 3),
        df.cve_id.iloc[j],
        df.description.iloc[j][:200]
    )

Saved files passed shape, ID alignment and finite-number checks.
1.007 CVE-2019-15706 An improper neutralization of input during web page generation in the SSL VPN portal of FortiProxy version 2.0.0, version 1.2.9 and below and FortiOS version 6.2.1 and below, version 6.0.8 and below, 
0.964 CVE-2024-26006 An improper neutralization of input during web page Generation vulnerability [CWE-79] in FortiOS version 7.4.3 and below, version 7.2.7 and below, version 7.0.13 and below and FortiProxy version 7.4.3
0.953 CVE-2025-31366 An Improper Neutralization of Input During Web Page Generation vulnerability [CWE-79] vulnerability in Fortinet FortiOS 7.6.0 through 7.6.3, FortiOS 7.4.0 through 7.4.8, FortiOS 7.2 all versions, Fort
0.953 CVE-2024-23111 An improper neutralization of input during web page Generation ('Cross-site Scripting') vulnerability [CWE-79] in FortiOS version 7.4.3 and below, 7.2 all versions, 7.0 all versions and FortiProxy ver
0.945 CVE-2025-54353 An Improper Neutralization

In [10]:
norms = np.linalg.norm(emb32, axis=1)
print("Vector norms (min / median / max):",
      norms.min(), np.median(norms), norms.max())

assert (norms > 0).all()
emb32 /= norms[:, None]

sims = emb32 @ emb32[i]
for j in np.argsort(-sims)[:6]:
    print(round(float(sims[j]), 3),
          df.cve_id.iloc[j],
          df.description.iloc[j][:200])

Vector norms (min / median / max): 0.99482816 1.0007514 1.0050033
1.0 CVE-2019-15706 An improper neutralization of input during web page generation in the SSL VPN portal of FortiProxy version 2.0.0, version 1.2.9 and below and FortiOS version 6.2.1 and below, version 6.0.8 and below, 
0.96 CVE-2024-26006 An improper neutralization of input during web page Generation vulnerability [CWE-79] in FortiOS version 7.4.3 and below, version 7.2.7 and below, version 7.0.13 and below and FortiProxy version 7.4.3
0.951 CVE-2025-31366 An Improper Neutralization of Input During Web Page Generation vulnerability [CWE-79] vulnerability in Fortinet FortiOS 7.6.0 through 7.6.3, FortiOS 7.4.0 through 7.4.8, FortiOS 7.2 all versions, Fort
0.947 CVE-2024-23111 An improper neutralization of input during web page Generation ('Cross-site Scripting') vulnerability [CWE-79] in FortiOS version 7.4.3 and below, 7.2 all versions, 7.0 all versions and FortiProxy ver
0.942 CVE-2021-26087 An improper neutralization o

In [11]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

print("Embedding model unloaded. Ready for the Granite language model.")

Embedding model unloaded. Ready for the Granite language model.


In [12]:
!pip -q install -U transformers accelerate

from transformers import AutoTokenizer, AutoModelForCausalLM

LLM = 'ibm-granite/granite-3.3-2b-instruct'

tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(
    LLM,
    torch_dtype=torch.float16,
    device_map='auto'
)
llm.eval()

print("Granite language model loaded.")
print("Device:", llm.device)

config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Granite language model loaded.
Device: cuda:0


In [13]:
def explain_cve(cve_id, description):
    messages = [{
        "role": "user",
        "content": (
            "You are a security engineer. Write exactly two short sentences "
            "on two lines, using only the supplied description. "
            "First explain what an attacker could do and any stated prerequisites. "
            "Then explain why the impact warrants patching. "
            "Do not invent active exploitation, public exploit code, scores, "
            "or claim this outranks other vulnerabilities.\n\n"
            f"CVE: {cve_id}\nDescription: {description}"
        )
    }]

    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(llm.device)

    with torch.inference_mode():
        output = llm.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    return tok.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()


results = []

for _, row in df.head(5).iterrows():
    torch.cuda.synchronize()
    start = time.perf_counter()

    explanation = explain_cve(row.cve_id, row.description)

    torch.cuda.synchronize()
    seconds = time.perf_counter() - start

    results.append({
        "cve_id": row.cve_id,
        "description": row.description,
        "explanation": explanation,
        "seconds": seconds
    })

    print(f"\n{row.cve_id} | {seconds:.1f} seconds")
    print(explanation)

pd.DataFrame(results).to_csv(
    OUT / "granite_rationale_test.csv", index=False
)

print("\nAverage seconds per CVE:",
      round(np.mean([r["seconds"] for r in results]), 1))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



CVE-2002-20002 | 3.7 seconds
1. An attacker could potentially predict or manipulate cryptographic keys generated by the Net::EasyTCP package, exploiting its use of Perl's non-secure rand() function.
2. This vulnerability warrants patching due to the risk of unauthorized access or data manipulation, as predictable keys could compromise the security and integrity of encrypted communications.

CVE-2003-20001 | 4.5 seconds
1. An attacker could exploit this vulnerability by sending a specially crafted TELNET request to a Mitel ICP VoIP 3100 device during the login wait time, while an external call is in progress.
2. The impact warrants patching because it allows unauthorized disclosure of sensitive call and SMDR (System Management Data Record) information, potentially exposing confidential communication details and violating privacy regulations.

CVE-2005-10003 | 5.1 seconds
An attacker could exploit CVE-2005-10003 by remotely manipulating the 'cmd' argument in mikexstudios Xcomic, leading

In [1]:
top60 = pd.read_csv(
    '/content/drive/MyDrive/FixFirst/top60_for_granite.csv',
    dtype=str
)

def why_fix_first(cve_id, description, signals):
    messages = [{
        "role": "user",
        "content": (
            f"You are a senior security engineer. In exactly two short "
            f"sentences, explain why {cve_id} should be prioritized for patching. "
            "First state concretely what an attacker gains and any stated "
            "prerequisites; then explain the priority using the supplied signals. "
            "Use only the supplied facts. Public exploit code does not by itself "
            "prove active exploitation. No generic advice or invented details.\n\n"
            f"Description: {description}\nKnown signals: {signals}"
        )
    }]

    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(llm.device)

    with torch.inference_mode():
        output = llm.generate(
            **inputs,
            max_new_tokens=90,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    return tok.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

for _, row in top60.head(3).iterrows():
    print(row.cve_id)
    print(why_fix_first(
        row.cve_id, row.description, row.top_reasons
    ))
    print()

NameError: name 'pd' is not defined

In [2]:
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')

print("Language model loaded:", 'llm' in globals())
print("Tokenizer loaded:", 'tok' in globals())

Mounted at /content/drive
Language model loaded: False
Tokenizer loaded: False


In [3]:
!pip -q install -U transformers accelerate

from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), (
    "Select Runtime → Change runtime type → T4 GPU first."
)

LLM = 'ibm-granite/granite-3.3-2b-instruct'

tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(
    LLM,
    dtype=torch.float16,
    device_map='auto'
)
llm.eval()

print("Granite loaded on:", llm.device)

config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/29.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Granite loaded on: cuda:0


In [4]:
top60 = pd.read_csv(
    '/content/drive/MyDrive/FixFirst/top60_for_granite.csv',
    dtype=str
)

def why_fix_first(cve_id, description, signals):
    messages = [{
        "role": "user",
        "content": (
            f"You are a senior security engineer. In exactly two short "
            f"sentences, explain why {cve_id} should be prioritized for patching. "
            "First state concretely what an attacker gains and any stated "
            "prerequisites; then explain the priority using the supplied signals. "
            "Use only the supplied facts. Public exploit code does not by itself "
            "prove active exploitation. No generic advice or invented details.\n\n"
            f"Description: {description}\nKnown signals: {signals}"
        )
    }]

    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(llm.device)

    with torch.inference_mode():
        output = llm.generate(
            **inputs,
            max_new_tokens=90,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    return tok.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

for _, row in top60.head(3).iterrows():
    print(row.cve_id)
    print(why_fix_first(
        row.cve_id, row.description, row.top_reasons
    ))
    print()


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FixFirst/top60_for_granite.csv'

In [6]:
from pathlib import Path

root = Path('/content/drive/MyDrive')

print("MY DRIVE:")
for p in root.iterdir():
    print(p.name)

print("\nTOP60 FILE LOCATIONS:")
matches = list(root.rglob('*top60*'))
for p in matches:
    print(p)

if not matches:
    print("No top60 file found in mounted My Drive.")

MY DRIVE:
Colab Notebooks
Getting started.pdf
Insert Title.gdoc
Bipolar, Sleeping Beauty & Case Study.gdoc
survey analysis.docx.gdoc
aps-ocd.gdoc
The Chosen People.gdoc
Step One: Research Proposal | 10%.gdoc
Untitled document (53).gdoc
Untitled document (52).gdoc
Untitled document (51).gdoc
The Quest for Balance to Gain Identity- Inside Out.docx.gdoc
Charts.gdoc
whatever.gdoc
inside out essay.gdoc
Volunteering Dilemma.gdoc
Hydrogen Peroxide Breakdown.gdoc
Overcoming the Boundaries.gdoc
Copy of inside out essay.gdoc
The Quest for Balance to Gain Identity- Inside Out.gdoc
Oedipus the King.gslides
5.2 Calorimeter and Enthalpy.gdoc
3.1 Metabolism and Energy.gdoc
Aristotle's Poetics.gdoc
Calorimetry and Enthalpy Lab Report.gdoc
EasyBib Bibliography: 10 25 2016 1:46 AM.gdoc
Unit 2: Freedom and Responsibility.gdoc
William Shakespeare's .gslides
Forgiveness Apology Letter.gdoc
Rate of Reaction.gdoc
UNICEF.gdoc
Unit 3: Sources of Moral Support.gdoc
From Manifest to Latent Messages.gdoc
Untitled

In [7]:
top60 = pd.read_csv(
    '/content/drive/MyDrive/FixFirst/top60_for_granite.csv',
    dtype=str
)

def why_fix_first(cve_id, description, signals):
    messages = [{
        "role": "user",
        "content": (
            f"You are a senior security engineer. In exactly two short "
            f"sentences, explain why {cve_id} should be prioritized for patching. "
            "First state concretely what an attacker gains and any stated "
            "prerequisites; then explain the priority using the supplied signals. "
            "Use only the supplied facts. Public exploit code does not by itself "
            "prove active exploitation. No generic advice or invented details.\n\n"
            f"Description: {description}\nKnown signals: {signals}"
        )
    }]

    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(llm.device)

    with torch.inference_mode():
        output = llm.generate(
            **inputs,
            max_new_tokens=90,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    return tok.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

for _, row in top60.head(3).iterrows():
    print(row.cve_id)
    print(why_fix_first(
        row.cve_id, row.description, row.top_reasons
    ))
    print()

CVE-2026-10523
CVE-2026-10523 poses a significant risk as it enables a remote unauthenticated attacker to create arbitrary administrative accounts, thereby gaining full administrative access to Ivanti Sentry versions R10.5.2, R10.6.2, and R10.7.1. The high-exposure vendor and public exploit code on GitHub, coupled with the low

CVE-2026-71362
CVE-2026-71362 poses a significant risk as it enables an attacker to gain unauthorized access to sensitive resources, bypassing intended access controls. The public exploit code available on GitHub, coupled with the high exposure of the vendor (Adobe) and the low EPSS score of 0.25, signifies a high priority for patching to prevent potential privilege escalation without requiring user interaction.

CVE-2026-2699
CVE-2026-2699 poses a significant risk as an unauthenticated attacker can exploit this vulnerability to access and modify restricted configuration pages, potentially leading to remote code execution. The presence of a public exploit code o

In [9]:
def why_fix_first(cve_id, description, signals):
    prompt = f"""You are a senior security engineer writing a patch-priority note for {cve_id}.
Write exactly two sentences, at most 40 words in total. Do not start with the CVE id.
Sentence 1: what an attacker gains and the prerequisite (e.g. unauthenticated, network access, user click).
Sentence 2: why it ranks high, citing only the supplied signals.
Scale: EPSS is the probability of exploitation within 30 days; 0.05 is high, 0.20+ is very high.
Public exploit code lowers attacker effort; it does NOT prove attacks are happening — never claim
monitoring, campaigns, or active exploitation. No generic advice, no invented details.
Do not end with a call to action such as "patch immediately"; end sentence 2 with the strongest signal itself.
Description: {description}
Known signals: {signals}"""

    inputs = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(llm.device)

    with torch.inference_mode():
        output = llm.generate(
            **inputs,
            max_new_tokens=90,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    return tok.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()


for _, row in top60.head(3).iterrows():
    print(row.cve_id)
    print(why_fix_first(
        row.cve_id, row.description, row.top_reasons
    ))
    print()

CVE-2026-10523
An attacker can exploit this Authentication Bypass vulnerability (CWE-288) to create arbitrary administrative accounts and gain full administrative access remotely without authentication, posing a significant risk due to the high EPSS score of 0.52 and the availability of a public exploit code on GitHub.

CVE-2026-71362
An attacker can exploit CVE-2026-71362 in Adobe Commerce to gain unauthorized elevated access, requiring no user interaction. The high EPSS score of 0.25, coupled with a publicly available exploit code on GitHub, underscores the critical need for immediate patching.

CVE-2026-2699
An unauthenticated attacker can exploit the vulnerability to access and modify restricted configuration pages, enabling potential remote code execution. The high EPSS score of 0.24, coupled with publicly available exploit code on GitHub, underscores the very high risk of exploitation.



In [10]:
rows = []

for _, row in top60.iterrows():
    rationale = why_fix_first(
        row.cve_id, row.description, row.top_reasons
    )
    rows.append({
        "cve_id": row.cve_id,
        "rationale": rationale
    })
    print(f"{len(rows)}/{len(top60)} completed")

rationales = pd.DataFrame(rows, columns=["cve_id", "rationale"])

output_path = '/content/drive/MyDrive/FixFirst/rationales.csv'
rationales.to_csv(output_path, index=False)

print("\nSaved:", output_path)

for position in [0, 1, 2, 57, 58, 59]:
    row = rationales.iloc[position]
    print(f"\nRow {position + 1}: {row.cve_id}")
    print(row.rationale)

1/60 completed
2/60 completed
3/60 completed
4/60 completed
5/60 completed
6/60 completed
7/60 completed
8/60 completed
9/60 completed
10/60 completed
11/60 completed
12/60 completed
13/60 completed
14/60 completed
15/60 completed
16/60 completed
17/60 completed
18/60 completed
19/60 completed
20/60 completed
21/60 completed
22/60 completed
23/60 completed
24/60 completed
25/60 completed
26/60 completed
27/60 completed
28/60 completed
29/60 completed
30/60 completed
31/60 completed
32/60 completed
33/60 completed
34/60 completed
35/60 completed
36/60 completed
37/60 completed
38/60 completed
39/60 completed
40/60 completed
41/60 completed
42/60 completed
43/60 completed
44/60 completed
45/60 completed
46/60 completed
47/60 completed
48/60 completed
49/60 completed
50/60 completed
51/60 completed
52/60 completed
53/60 completed
54/60 completed
55/60 completed
56/60 completed
57/60 completed
58/60 completed
59/60 completed
60/60 completed

Saved: /content/drive/MyDrive/FixFirst/rationale

In [2]:
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip -q install -U transformers accelerate

from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), (
    "Select Runtime → Change runtime type → T4 GPU first."
)

LLM = 'ibm-granite/granite-3.3-2b-instruct'
tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(
    LLM,
    dtype=torch.float16,
    device_map='auto'
)
llm.eval()

print("Granite loaded on:", llm.device)

config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/29.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Granite loaded on: cuda:0


In [4]:
def why_fix_first(cve_id, description, signals):
    prompt = f"""You are a senior security engineer writing a patch-priority note for {cve_id}.
Write exactly two sentences, at most 40 words in total. Do not start with the CVE id.
Sentence 1: what an attacker gains and the prerequisite (e.g. unauthenticated, network access, user click).
Sentence 2: why it ranks high, citing only the supplied signals.
Scale: EPSS is the probability of exploitation within 30 days; 0.05 is high, 0.20+ is very high.
Public exploit code lowers attacker effort; it does NOT prove attacks are happening — never claim
monitoring, campaigns, or active exploitation. No generic advice, no invented details.
Do not end with a call to action such as "patch immediately"; end sentence 2 with the strongest signal itself.
Description: {description}
Known signals: {signals}"""

    inputs = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(llm.device)

    with torch.inference_mode():
        output = llm.generate(
            **inputs,
            max_new_tokens=90,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    return tok.decode(
        output[0, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

In [5]:
import re

missing = pd.read_csv(
    '/content/drive/MyDrive/FixFirst/top100_missing_for_granite.csv',
    dtype=str
)

assert len(missing) == 55, f"Expected 55 rows, found {len(missing)}"

rows = []

for _, row in missing.iterrows():
    signals = (
        f"EPSS {row.epss}; "
        f"{row.top_reasons if pd.notna(row.top_reasons) else ''}"
    )

    rationale = why_fix_first(
        row.cve_id, row.description, signals
    )

    # Trim unfinished trailing text; don't mistake decimal points
    # in EPSS scores or version numbers for sentence endings.
    endings = list(re.finditer(r'[.!?](?=\s|$)', rationale))
    if endings:
        rationale = rationale[:endings[-1].end()].strip()

    rows.append({
        "cve_id": row.cve_id,
        "rationale": rationale
    })
    print(f"{len(rows)}/{len(missing)} completed")

rationales_2 = pd.DataFrame(rows, columns=["cve_id", "rationale"])
output_path = '/content/drive/MyDrive/FixFirst/rationales_2.csv'
rationales_2.to_csv(output_path, index=False)

print("\nSaved:", output_path)

for position in [0, 1, len(rationales_2) - 1]:
    row = rationales_2.iloc[position]
    print(f"\nRow {position + 1}: {row.cve_id}")
    print(row.rationale)

1/55 completed
2/55 completed
3/55 completed
4/55 completed
5/55 completed
6/55 completed
7/55 completed
8/55 completed
9/55 completed
10/55 completed
11/55 completed
12/55 completed
13/55 completed
14/55 completed
15/55 completed
16/55 completed
17/55 completed
18/55 completed
19/55 completed
20/55 completed
21/55 completed
22/55 completed
23/55 completed
24/55 completed
25/55 completed
26/55 completed
27/55 completed
28/55 completed
29/55 completed
30/55 completed
31/55 completed
32/55 completed
33/55 completed
34/55 completed
35/55 completed
36/55 completed
37/55 completed
38/55 completed
39/55 completed
40/55 completed
41/55 completed
42/55 completed
43/55 completed
44/55 completed
45/55 completed
46/55 completed
47/55 completed
48/55 completed
49/55 completed
50/55 completed
51/55 completed
52/55 completed
53/55 completed
54/55 completed
55/55 completed

Saved: /content/drive/MyDrive/FixFirst/rationales_2.csv

Row 1: CVE-2025-69971
Attackers can forge admin tokens and gain full ad